# Gemini Agent Session Inference Manifest Generation

This Colab notebook prepares Gemini's session-aware transcription manifests for evaluation by merging predictions from our Vertex AI Agent Sessions pipeline with ground-truth segmentations.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/watch-duty/radio-transcription/blob/main/model/colabs/gemini_agent_session/create_inference_manifest_masked_audio.ipynb)

### Key Differences from the Standard Batch Manifest Tool

Unlike `model/colabs/gemini_create_inference_manifest.ipynb` which operates on Vertex Batch Prediction outputs, this notebook is custom-tailored to support the context-retaining Agent Sessions pipeline:
1. **Agent Sessions Output Parser**: Parses predictions directly from the clean NDJSON checkpoints created by our session transcription loop (`{"audio_filepath": "...", "transcript": "..."}`), avoiding complex nested Vertex Batch JSON schemas.
2. **Direct Segment Key Matching**: Rather than matching records using floating-point timestamps (which are prone to precision/rounding errors), this notebook uses direct `(example_id, segment_id)` lookups parsed directly from the segment FLAC filenames, ensuring **100% match accuracy**.

### Core Functions

1. **Loads Ground Truth Manifest**: Pulls the baseline ASR manifest containing segment information from GCS.
2. **Collects Session Predictions**: Downloads and aggregates session checkpoint `.jsonl` outputs from the configured GCS results folder.
3. **Resolves Exact Key Matches**: Extracts `example_id` (channel) and `segment_id` from filenames and maps transcripts directly to their target segment rows.
4. **Generates Labeled Manifest**: Outputs a unified `.jsonl` file matching the evaluation framework schema.
5. **Automated Export & Preview**: Backs up the finalized manifest to GCS and displays a premium, structured tabular preview of the merged transcriptions.


In [ ]:
# @title Bootstrap and Install Environment
import os

# Clone repository if not already present
if not os.path.exists("radio-transcription"):
    !git clone -q https://github.com/watch-duty/radio-transcription.git

# Install the model library in editable mode
try:
    import common

    print("✅ Library 'common' already installed.")
except ImportError:
    print("Installing library and dependencies...")
    %pip install -q -e radio-transcription/model loguru

    import site
    import importlib

    importlib.reload(site)

    print("\n✅ Dependencies installed successfully.")

In [ ]:
# @title Imports
import collections
import json
import re
from pathlib import Path
from typing import Any

from google.cloud import storage
from google.colab import auth, userdata
from IPython.display import display
from loguru import logger
import pandas as pd

from common.manifest import load_manifest

In [ ]:
# @title Authentication and client initialization
print("Attempting standard browser authentication...")
auth.authenticate_user()
print("Browser authentication successful!")

GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")
!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title Pipeline Configuration
# fmt: off
# @markdown ### Input/Output Configuration
# @markdown Partial path under `gs://{GCS_BUCKET}/segmented_audio/` where `batch_manifest.jsonl` and audio segments are located (e.g., `broadcastify/calls/eval_audio_masked_v2`).
INPUT_AUDIO_DIR = "broadcastify/calls/eval_audio_masked_v2"  # @param {type:"string"}
# @markdown Partial path under `gs://{GCS_BUCKET}/transcripts/` where outputs will be stored (e.g., `broadcastify/calls/eval`).
OUTPUT_TRANSCRIPT_DIR = "broadcastify/calls/eval"  # @param {type:"string"}
# @markdown Experiment Name (representing the subfolder under the model directory, e.g., bcfy_calls_v1)
EXPERIMENT_NAME = "bcfy_calls_v1"  # @param {type:"string"}

# @markdown ### Model Selection (Matches transcribe_masked_audio config)
# @markdown **Option A: Select a base model**
MODEL_ID = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.1-pro-preview", "gemini-3.5-flash"] {type:"string"}
# @markdown **Option B: Use a Tuned ML Model (Deployed in multi-region us)**
USE_CUSTOM_MODEL = False  # @param {type:"boolean"}
# fmt: on

assert INPUT_AUDIO_DIR, "INPUT_AUDIO_DIR must be provided and cannot be empty."
assert OUTPUT_TRANSCRIPT_DIR, (
    "OUTPUT_TRANSCRIPT_DIR must be provided and cannot be empty."
)
assert MODEL_ID, "MODEL_ID must be provided and cannot be empty."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided and cannot be empty."

if USE_CUSTOM_MODEL:
    GCP_CUSTOM_MODEL_NAME = userdata.get("GCP_CUSTOM_MODEL_NAME")
    assert GCP_CUSTOM_MODEL_NAME, (
        "GCP_CUSTOM_MODEL_NAME must be provided in Colab userdata."
    )
    MODEL_ID_DIR = GCP_CUSTOM_MODEL_NAME
else:
    MODEL_ID_DIR = re.sub(r"[-\.]", "_", MODEL_ID)

INPUT_MANIFEST_PATH = f"segmented_audio/{INPUT_AUDIO_DIR}/batch_manifest.jsonl"
INPUT_TRANSCRIPTS_PATH = (
    f"transcripts/{OUTPUT_TRANSCRIPT_DIR}/{MODEL_ID_DIR}/{EXPERIMENT_NAME}"
)
OUTPUT_MANIFEST_PATH = f"inference_manifests/{OUTPUT_TRANSCRIPT_DIR}/{MODEL_ID_DIR}/{EXPERIMENT_NAME}.jsonl"

MODEL_VERSION = re.sub(r"[-\.]", "_", MODEL_ID_DIR)

In [ ]:
# @title Download batch manifest
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"
!gcloud storage cp gs://{GCS_BUCKET}/{INPUT_MANIFEST_PATH} ./{BATCH_MANIFEST_FILENAME}

In [ ]:
# @title Manifest Processing Functions
def merge_gcs_results_to_manifest(
    batch_manifest_data: list[dict[str, Any]],
    gcs_bucket_name: str,
    output_file: str,
) -> dict[str, Any]:
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(gcs_bucket_name)

    gemini_predictions = {}

    # Try to find the results folder
    prefix = f"{INPUT_TRANSCRIPTS_PATH}/"
    logger.info(
        f"Searching for predictions with prefix: gs://{gcs_bucket_name}/{prefix}"
    )

    blobs = list(bucket.list_blobs(prefix=prefix))
    # Only include files named exactly 'predictions.jsonl'
    valid_blobs = [
        b
        for b in blobs
        if b.name.split("/")[-1] == "predictions.jsonl" and (b.size or 0) > 0
    ]

    if not valid_blobs:
        logger.error(f"No files named 'predictions.jsonl' found in {prefix}")
        return {
            "total": len(batch_manifest_data),
            "matched": 0,
            "missing": len(batch_manifest_data),
        }

    logger.info(f"Found {len(valid_blobs)} valid 'predictions.jsonl' file(s).")
    for blob in valid_blobs:
        local_preds = "temp_preds.jsonl"
        blob.download_to_filename(local_preds)

        with open(local_preds) as f:
            for line in f:
                if not line.strip():
                    continue
                try:
                    data = json.loads(line)
                    audio_uri = data.get("audio_filepath", "")
                    if not audio_uri:
                        continue

                    filename = Path(audio_uri).stem
                    if "__seg" not in filename:
                        continue

                    example_id, seg_key = filename.split("__seg")
                    raw_text = data.get("transcript", "").strip()
                    gemini_predictions[(example_id, seg_key)] = raw_text
                except Exception as e:
                    continue

    merged_records = []
    matched_count = 0
    for b_info in batch_manifest_data:
        gemini_text = ""
        example_id = b_info.get("example_id", "")
        segment_id = b_info.get("segment_id", "")
        if (example_id, segment_id) in gemini_predictions:
            gemini_text = gemini_predictions[(example_id, segment_id)]
            matched_count += 1

        merged_records.append(
            {**b_info, f"pred_text_{MODEL_VERSION}": gemini_text}
        )

    with open(output_file, "w", encoding="utf-8") as f_out:
        f_out.writelines(json.dumps(rec) + "\n" for rec in merged_records)

    return {
        "total": len(batch_manifest_data),
        "matched": matched_count,
        "missing": len(batch_manifest_data) - matched_count,
    }

In [ ]:
# @title Merge GCS Predictions & Upload Manifest
LOCAL_INFERENCE_MANIFEST = "inference_manifest.jsonl"
stats = merge_gcs_results_to_manifest(
    batch_manifest_data=load_manifest(BATCH_MANIFEST_FILENAME),
    gcs_bucket_name=GCS_BUCKET,
    output_file=LOCAL_INFERENCE_MANIFEST,
)

logger.info(
    f"Processing complete: {stats['matched']} matches found out of {stats['total']} total segments."
)

if stats["matched"] > 0:
    # Upload the final manifest back to GCS
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)
    bucket.blob(OUTPUT_MANIFEST_PATH).upload_from_filename(
        LOCAL_INFERENCE_MANIFEST
    )
    logger.info(
        f"Successfully uploaded merged manifest to: gs://{GCS_BUCKET}/{OUTPUT_MANIFEST_PATH}"
    )

    # Premium Visual Receipt Table

    merged_data = load_manifest(LOCAL_INFERENCE_MANIFEST)
    if merged_data:
        df = pd.DataFrame(merged_data)
        display(
            df[["example_id", "segment_id", f"pred_text_{MODEL_VERSION}"]].head(
                10
            )
        )